<div style="background-color: #f0f0f0; color:rgb(56, 56, 56); padding: 8px; display: flex; align-items: center; gap: 100px;">
  <img src="../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI<br/>
    3.Lab.1 RNN Implementation for Audio Classification<br/>
  </strong>
  <emph>
    Student Name: [Complete with your name]<br/>
    Date: [Fill up with the submission date]<br/>
  </emph>
</div>

## Background

This hands-on lab will guide you through implementing RNN models for audio classification. You'll work with a dataset of spoken commands and explore how different RNN architectures perform on this task. By the end of this lab, you'll understand how RNNs process sequential audio data and how to build, train, and evaluate them using PyTorch.

---

## Description

During this lab session, you will:

- Load and preprocess audio data (waveforms to spectrograms/MFCCs)
- Implement a vanilla RNN model using PyTorch
- Train and evaluate the model on a spoken command classification task
- Explore the impact of sequence length and feature extraction methods
- Visualize RNN hidden state dynamics during inference
- Analyze model performance on different types of audio inputs
- Compare results with baseline non-sequential models

**Key Deliverables:**
- Complete the provided notebook with all code cells executed
- Working implementation of an RNN for audio classification
- Visualizations of model performance and hidden states
- Answer the embedded questions about RNN behavior with audio data
- Submit your notebook with all cells executed and outputs visible

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ⚠️ IMPORTANT NOTICE FOR STUDENTS ⚠️
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchaudio
import torchaudio.transforms as T
from torchaudio.datasets import SPEECHCOMMANDS

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tqdm import tqdm

import os
import warnings

warnings.filterwarnings("ignore")

from rich.console import Console
from rich.table import Table

console = Console()

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
console.print(f"Using device: {device}")

## Part 1: Data Loading and Exploration

### 1.1 Load the Speech Commands Dataset

We'll use the **Speech Commands dataset** which contains short audio clips of spoken words. This is a common benchmark for audio classification tasks.

**Dataset Info:**
- 35 different command words
- ~1 second audio clips
- 16kHz sample rate
- Multiple speakers per command

In [ ]:
# Define the subset of commands we want to classify
COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]


class SubsetSC(SPEECHCOMMANDS):
    """Filters the Speech Commands dataset to only include specified labels."""

    def __init__(self, subset: str = None, commands: list = None):
        super().__init__(
            "/content/drive/MyDrive/Laboratories/data/", download=True, subset=subset
        )

        if commands:

            def load_list(filename):
                filepath = os.path.join(self._path, filename)
                with open(filepath) as fileobj:
                    return [
                        os.path.normpath(os.path.join(self._path, line.strip()))
                        for line in fileobj
                    ]

            if subset == "validation":
                val_list = load_list("validation_list.txt")
                self._walker = [w for w in self._walker if w in val_list]
                pass
            elif subset == "testing":
                testing_list = load_list("testing_list.txt")
                self._walker = [w for w in self._walker if w in testing_list]
                pass
            elif subset == "training":
                excludes = load_list("validation_list.txt") + load_list(
                    "testing_list.txt"
                )
                excludes = set(excludes)
                self._walker = [w for w in self._walker if w not in excludes]

            self._walker = [w for w in self._walker if w.split(os.sep)[-2] in commands]


# Load train, validation, and test sets
train_set = SubsetSC(subset="training", commands=COMMANDS)
val_set = SubsetSC(subset="validation", commands=COMMANDS)
test_set = SubsetSC(subset="testing", commands=COMMANDS)

console.print(f"Training samples: {len(train_set)}")
console.print(f"Validation samples: {len(val_set)}")
console.print(f"Test samples: {len(test_set)}")
console.print(f"\nCommands to classify: {COMMANDS}")

### 1.2 Explore the Audio Data

In [ ]:
# Load and visualize a sample audio file
sample_waveform, sample_rate, sample_label, *_ = train_set[0]

console.print(f"Waveform shape: {sample_waveform.shape}")
console.print(f"Sample rate: {sample_rate} Hz")
console.print(f"Label: {sample_label}")
console.print(f"Duration: {sample_waveform.shape[1] / sample_rate:.2f} seconds")

plt.figure(figsize=(12, 4))
wave = sample_waveform[0]
time = np.arange(len(wave)) / sample_rate
plt.plot(time, wave)
plt.title(f"Waveform - Label: {sample_label}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.show()

#### REFLECTION QUESTION 1:

Why is audio data considered sequential data? What makes RNNs suitable for processing audio compared to feedforward networks?

**Your Answer:**

Audio is considered sequential data because it is composed of a series of patterns or samples ordered sequentially in time. Here, the order is really important for understanding the context of the audio. As for why RNNs are more suitable for processing audio compared to feedforward networks, I would think it is because RNNs were designed to work with data where order matters. An RNN can handle long or short sequences or input data step by step, while a fully connected (feedforward) network requires fixed-size inputs.

## Part 2: Feature Extraction

Raw audio waveforms are not ideal inputs for neural networks. We'll convert them to MFCCs (Mel-Frequency Cepstral Coefficients), which are more informative representations.

In [ ]:
mfcc_transform = T.MFCC(
    sample_rate=16000,
    n_mfcc=40,
    melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 64},
)

# Test the transform
features = mfcc_transform(sample_waveform)
console.print(f"Feature shape: {features.shape}")
console.print(f"Feature dimensions: [channels, n_mfcc, time_steps]")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Original waveform
wave = sample_waveform[0]
time = np.arange(len(wave)) / sample_rate
axes[0].plot(time, wave)
axes[0].set_xlabel("Time (seconds)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"Original Waveform - Label: {sample_label}")
axes[0].grid(True)

# Plot 2: MFCC features (use imshow)
im = axes[1].imshow(features[0].numpy(), aspect="auto", origin="lower", cmap="turbo")
axes[1].set_xlabel("Time Frames")
axes[1].set_ylabel("MFCC Coefficients")
axes[1].set_title("MFCC Features")
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

#### REFLECTION QUESTION 2:

What information do MFCCs capture from the audio signal? Why are they useful for audio classification?

**Your Answer:**

What I see is that MFCC captures audio signals, which are like the tones and shapes of sound at each moment. With the second diagram, we can see how at certain points the sound can be neutral or more intense, which I relate to how much energy there is. As for why they are useful for audio classification, it is because, thanks to the way it analyzes sound, the neural network can identify patterns that are useful for differentiating it from the rest. For example, here the audio labeled with “down” is being validated, and it will have patterns that will differentiate it from the rest.

### 2.1 Create Custom Dataset Class

We need to create a dataset class that applies feature extraction and handles variable-length sequences.

In [ ]:
# TODO: Implement the AudioDataset class
class AudioDataset(Dataset):
    def __init__(self, subset_dataset, transform, label_to_idx, max_length=101):
        """
        Args:
            subset_dataset: The Speech Commands dataset subset
            transform: Feature extraction transform (MFCC)
            label_to_idx: Dictionary mapping label strings to indices
            max_length: Maximum sequence length (time steps)
        """
        self.dataset = subset_dataset
        self.transform = transform
        self.label_to_idx = label_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample_waveform, sample_rate, sample_label, *_ = self.dataset[idx]

        features = self.transform(waveform).squeeze(0).T

        if features.shape[0] < self.max_length:
            pad_size = self.max_length - features.shape[0]
            padding = torch.zeros(pad_size, features.shape[1])
            features = torch.cat([features, padding], dim=0)
        else:
            features = features[: self.max_length, :]

        label_idx = self.label_to_idx[sample_label]

        return features, label_idx


# Create label-to-index mapping
label_to_idx = {label: idx for idx, label in enumerate(COMMANDS)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

console.print(f"Label mapping: {label_to_idx}")

In [ ]:
train_dataset = AudioDataset(train_set, mfcc_transform, label_to_idx)
val_dataset = AudioDataset(val_set, mfcc_transform, label_to_idx)
test_dataset = AudioDataset(test_set, mfcc_transform, label_to_idx)

batch_size = 32
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

console.print(f"Number of training batches: {len(train_loader)}")
console.print(f"Number of validation batches: {len(val_loader)}")
console.print(f"Number of test batches: {len(test_loader)}")

## Part 3: RNN Model Implementation

Now let's implement a vanilla RNN model for audio classification.

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        """
        Args:
            input_size: Number of input features (e.g., 40 for MFCC)
            hidden_size: Number of hidden units in RNN
            num_layers: Number of stacked RNN layers
            num_classes: Number of output classes
            dropout: Dropout probability
        """
        super(VanillaRNN, self).__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)

        self.hidden_size = hidden_size
        self.num_layers = num_layers

    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (batch_size, seq_length, input_size)

        Returns:
            out: Output logits of shape (batch_size, num_classes)
            hidden: Final hidden state for visualization
        """
        out, hidden = self.rnn(x)

        out = out[:, -1, :]

        out = self.dropout(out)

        out = self.fc(out)

        return out, hidden


# Instantiate the model
input_size = 40  # Number of MFCC coefficients
hidden_size = 128
num_layers = 2
num_classes = len(COMMANDS)

model = VanillaRNN(input_size, hidden_size, num_layers, num_classes).to(device)

console.print(model)
console.print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

#### REFLECTION QUESTION 3:

Explain the role of the hidden state in an RNN. How does it allow the network to process sequential data?

**Your Answer:**

What I observed is that the hidden state functions as the network's memory. Each time it processes an element of the sequence, it stores relevant information about what it has seen so far. When the next element arrives, the RNN uses this memory together with the new input to generate its output, allowing it to remember context and dependencies throughout the sequence. Thanks to this, the RNN understands the entire sequence.

## Part 4: Training

Let's train our RNN model on the audio classification task.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # 1. Iterate through batches
    for features, labels in tqdm(train_loader, desc="Training"):
        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / len(train_loader), correct / total


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # Similar to training but without gradient updates
    with torch.no_grad():
        for features, labels in tqdm(val_loader, desc="Validation"):
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)

            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(val_loader), correct / total

In [ ]:
# Train the model
num_epochs = 20
train_losses = []
train_accs = []
val_losses = []
val_accs = []

best_val_acc = 0.0

for epoch in range(num_epochs):
    console.print(f"\nEpoch {epoch+1}/{num_epochs}")

    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Update learning rate
    scheduler.step(val_loss)

    # Print metrics
    console.print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    console.print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_rnn_model.pth")
        console.print(f"Best model saved with validation accuracy: {val_acc:.4f}")

console.print(f"\nBest validation accuracy: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(
    range(1, len(train_losses) + 1), train_losses, label="Training Loss", marker="o"
)
axes[0].plot(
    range(1, len(val_losses) + 1), val_losses, label="Validation Loss", marker="s"
)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(
    range(1, len(train_accs) + 1), train_accs, label="Training Accuracy", marker="o"
)
axes[1].plot(
    range(1, len(val_accs) + 1), val_accs, label="Validation Accuracy", marker="s"
)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training and Validation Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### REFLECTION QUESTION 4:

Analyze the training curves. Is there evidence of overfitting or underfitting? What could you do to improve the model?

**Your Answer:**

By analyzing the training curves, the model appears to be overfitting. The training loss keeps decreasing and the training accuracy improves steadily, but the validation curves do not follow the same trend: the validation loss stops improving early, and the validation accuracy grows much more slowly. This suggests that the model is learning patterns specific to the training set instead of generalizing well. To improve this, we could add more regularization (e.g., dropout or weight decay), apply data augmentation on the audio such as adding noise or time-shifting, or simplify the model so it has fewer parameters. Additionally, tuning hyperparameters like the learning rate or training for fewer epochs could also help reduce overfitting.

## Part 5: Evaluation

Let's evaluate the trained model on the test set.

In [ ]:
model.load_state_dict(torch.load("best_rnn_model.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        target = target.to(device)

        output, _ = model(data)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(target.cpu().numpy())

# Calculate metrics
test_accuracy = accuracy_score(all_labels, all_preds)
console.print(f"Test Accuracy: {test_accuracy:.4f}")
console.print("\nClassification Report:")
console.print(classification_report(all_labels, all_preds, target_names=COMMANDS))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", xticklabels=COMMANDS, yticklabels=COMMANDS
)
plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

#### REFLECTION QUESTION 5:

Looking at the confusion matrix, which commands does the model confuse most often? Why might this be the case?

**Your Answer:**

When reviewing the confusion matrix, the commands that the model most often confuses are those that sound similar or have very similar acoustic patterns. For example, short words such as "up", "off", "on", "go", or "no" tend to get mixed up because they share very short vowels and similar initial consonants. In general, these confusions occur because the model does not yet fully grasp the subtle differences in sound, either due to a lack of data, variability in voices, or because the words are actually very similar acoustically.

## Part 6: Hidden State Visualization

Let's visualize how the RNN's hidden state evolves as it processes an audio sequence.

In [ ]:
def visualize_hidden_states(model, sample_input, label, device):
    """
    Visualize RNN hidden state dynamics during processing.

    Args:
        model: Trained RNN model
        sample_input: Input features (seq_length, input_size)
        label: True label of the sample
        device: Device to run on
    """

    x = sample_input.unsqueeze(0).to(device)

    output, hidden = model.rnn(x)

    hidden_over_time = output[0].detach().cpu().numpy()

    plt.figure(figsize=(12, 6))
    sns.heatmap(
        hidden_over_time.T, cmap="viridis", cbar=True, xticklabels=10, yticklabels=False
    )
    plt.title(f"Hidden State Dynamics (Last Layer)\nTrue Label: {label}")
    plt.xlabel("Time Step")
    plt.ylabel("Hidden Units")
    plt.tight_layout()
    plt.show()


# Get a sample and visualize
sample_idx = 0
sample_input, sample_label = test_dataset[sample_idx]
visualize_hidden_states(model, sample_input, idx_to_label[sample_label], device)

## Part 7: Comparison with Baseline

Let's compare our RNN with a simple feedforward network to demonstrate the advantage of sequential modeling.

In [ ]:
class FeedforwardBaseline(nn.Module):
    def __init__(self, input_size, seq_length, hidden_size, num_classes):
        super(FeedforwardBaseline, self).__init__()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(seq_length * input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
baseline_model = FeedforwardBaseline(
    input_size=40, seq_length=101, hidden_size=128, num_classes=len(COMMANDS)
).to(device)

baseline_criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)
baseline_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    baseline_optimizer, mode="min", factor=0.5, patience=3
)

console.print(baseline_model)
console.print(
    f"Baseline parameters: {sum(p.numel() for p in baseline_model.parameters()):,}"
)

In [ ]:
num_epochs = 20
baseline_train_losses, baseline_train_accs = [], []
baseline_val_losses, baseline_val_accs = [], []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_epoch(
        baseline_model, train_loader, baseline_criterion, baseline_optimizer, device
    )
    val_loss, val_acc = validate(baseline_model, val_loader, baseline_criterion, device)

    baseline_train_losses.append(tr_loss)
    baseline_train_accs.append(tr_acc)
    baseline_val_losses.append(val_loss)
    baseline_val_accs.append(val_acc)

    baseline_scheduler.step(val_loss)

    console.print(
        f"[Baseline] Epoch {epoch+1:02d} | "
        f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.3f} | "
        f"Val   Loss: {val_loss:.4f} Acc: {val_acc:.3f}"
    )

In [ ]:
def test_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x) if isinstance(model, FeedforwardBaseline) else model(x)[0]
            _, pred = logits.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return correct / total


rnn_test_acc = test_accuracy(model, test_loader, device)
baseline_test_acc = test_accuracy(baseline_model, test_loader, device)

In [ ]:
table = Table(
    title="FINAL COMPARISON (Test set)", show_header=True, header_style="bold magenta"
)

table.add_column("Model", style="cyan", justify="left")
table.add_column("Test Accuracy", style="green", justify="center")
table.add_column("Parameters", style="yellow", justify="right")

table.add_row(
    "RNN", f"{rnn_test_acc:.4f}", f"{sum(p.numel() for p in model.parameters()):,}"
)
table.add_row(
    "Feed-Forward",
    f"{baseline_test_acc:.4f}",
    f"{sum(p.numel() for p in baseline_model.parameters()):,}",
)

console.print(table)

#### REFLECTION QUESTION 6:

Compare the performance of the RNN with the feedforward baseline. What does this tell you about the importance of modeling sequential dependencies in audio data?

**Your Answer:**

When comparing RNNs with feedforward models, RNNs typically perform better. This is because RNNs understand audio as a sequence, while feedforward models only see all the values flattened, as if they were a still image. When the model does not take into account the order of the samples, as in feedforward, it loses a lot of important sound information. In contrast, RNN can follow the rhythm of the audio and capture how it changes over time.

## Part 8: Analysis and Reflection

#### REFLECTION QUESTION 7:

What are the main challenges you encountered when working with audio data and RNNs? How did you address them?

**Your Answer:**

The main challenges I encountered when working with audio and RNNs were more practical than technical. The first was training: when using the free version of Google Colab, the models took a long time and I often lost progress due to usage limits. To solve this, I ended up purchasing processing units; although they weren't much faster, at least they allowed me to complete the training without interruptions.

Another challenge was understanding and completing some of the TODOs in the code, especially those related to the training process. I still don't have a good grasp of some concepts, so I had to rely on examples from Kaggle and compare different implementations to clarify my doubts.

#### REFLECTION QUESTION 8:

Based on your experiments, what hyperparameters had the most impact on model performance? What changes would you make to improve the model further?

**Your Answer:**

In my tests, what most affected the model's performance were things like the internal size of the RNN and the learning rate. When those values were not well adjusted, the model learned too little or learned too much and became confused. I also noticed that using 40 audio features (the MFCCs) helped the model understand sounds better.

If I wanted to improve it, I would try changing these values a little until I found a balance, neither too large nor too small. I would also use a little more care to prevent the model from overtraining, for example by increasing the dropout.

#### REFLECTION QUESTION 9:

Discuss potential applications of RNN-based audio classification in real-world scenarios. What additional challenges might arise in production environments?

**Your Answer:**

An RNN-based model for classifying audio could be used in many real-world situations, such as voice assistants, systems that activate functions with simple commands ("turn on", "turn off"), apps for people with disabilities, or devices that need to react to specific sounds.

In a production environment, new challenges arise: ambient noise can have a significant impact, people do not all speak the same way (accent, speed, tone), and microphones are not always of good quality. In addition, the system would have to work quickly and without errors, even with many users at the same time. All of this means that the model needs more adjustments, more real data, and an infrastructure that supports its daily use.

---

## Conclusion

Congratulations! You've successfully:
- Loaded and preprocessed audio data
- Implemented feature extraction using MFCCs
- Built a vanilla RNN model for audio classification
- Trained and evaluated the model
- Visualized hidden state dynamics
- Compared with a baseline model

### Next Steps

In Lab 2, you'll explore advanced RNN variants (LSTM and GRU) and see how they address the limitations of vanilla RNNs, particularly for longer sequences.